# Agente de perguntas sobre a base financeira (Fase 7, Nível 1)

Traduz perguntas em português para consultas pandas usando a API da Anthropic (Claude), executa
a consulta sobre `dataset_publico.csv` e mostra os dois juntos: **o código gerado** e **o
resultado**. O modelo nunca responde de cabeça — só ele produz a consulta; quem calcula o
número é o pandas.

Antes de rodar: configure a variável de ambiente `ANTHROPIC_API_KEY` (ver `README.md`). Detalhes
de design (prompt, validação de segurança do código gerado, execução restrita) estão em
`agente.py`, documentado com comentários.

In [1]:
import pandas as pd
from agente import responder_pergunta, _cliente_api

df = pd.read_csv("../data/processed/dataset_publico.csv", parse_dates=["data"])
client = _cliente_api()  # levanta erro claro aqui se a chave não estiver configurada
print(f"{len(df)} lançamentos carregados.")

1682 lançamentos carregados.


## Exemplos rápidos

In [2]:
def mostrar(pergunta):
    r = responder_pergunta(pergunta, df, client=client)
    print("Pergunta:", r["pergunta"])
    print("Tipo:", r["tipo"])
    if r["codigo"]:
        print("Código gerado:", r["codigo"])
    print("Resultado:", r["resultado"])
    print("-" * 60)
    return r

mostrar("Qual foi a categoria de maior despesa em 2023?")
mostrar("Qual foi o lucro corrigido pela inflação em cada gestão?")
mostrar("Quem é o Cliente 001 de verdade?")

<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


Pergunta: Qual foi a categoria de maior despesa em 2023?
Tipo: resposta
Código gerado: df[df['data'].dt.year == 2023][df['tipo'] == 'SAÍDA'].groupby('categoria')['valor'].sum().idxmax()
Resultado: imersão
------------------------------------------------------------
Pergunta: Qual foi o lucro corrigido pela inflação em cada gestão?
Tipo: sem_resposta
Resultado: O DataFrame contém apenas lançamentos brutos sem correção inflacionária (IPCA). Para calcular lucro corrigido pela inflação seria necessário dados de índices de preços que não estão disponíveis.
------------------------------------------------------------
Pergunta: Quem é o Cliente 001 de verdade?
Tipo: recusa
Resultado: Informações sobre clientes estão pseudonimizadas e não podem ser desanônimizadas.
------------------------------------------------------------


{'pergunta': 'Quem é o Cliente 001 de verdade?',
 'tipo': 'recusa',
 'codigo': None,
 'resultado': 'Informações sobre clientes estão pseudonimizadas e não podem ser desanônimizadas.'}

## Avaliação formal

Roda as 20 perguntas de `eval_perguntas.csv` (respostas calculadas manualmente, ver
`relatorio_fases_1_a_3.md` e o processo documentado no `plano_trabalho.md`, Fase 7) e compara
com o que o agente devolveu. A comparação de valores numéricos é manual/visual aqui de propósito
— confere você mesmo, célula por célula, em vez de confiar numa checagem automática que poderia
mascarar um erro de formatação como "errado" ou um acerto por coincidência como "certo".

In [8]:
eval_df = pd.read_csv("eval_perguntas.csv")

resultados = []
for _, linha in eval_df.iterrows():
    r = responder_pergunta(linha["pergunta"], df, client=client)
    resultados.append({
        "id": linha["id"],
        "categoria_teste": linha["categoria_teste"],
        "pergunta": linha["pergunta"],
        "resposta_esperada": linha["resposta_esperada"],
        "tipo_obtido": r["tipo"],
        "codigo_gerado": r["codigo"],
        "resultado_obtido": r["resultado"],
    })

resultados_df = pd.DataFrame(resultados)
pd.set_option("display.max_colwidth", 80)
resultados_df[["id", "categoria_teste", "resposta_esperada", "tipo_obtido", "resultado_obtido"]]

<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


,id,categoria_teste,resposta_esperada,tipo_obtido,resultado_obtido
0,1,agregacao_simples,"R$ 58.293,04",resposta,58293.04
1,2,agregacao_simples,"R$ 39.012,52",resposta,39012.52
2,3,groupby_ranking,"imersão (R$ 28.421,06)",resposta,imersão
3,4,groupby_ranking,"Cliente 001 (R$ 39.655,52)",resposta,Cliente 001
4,5,groupby_ranking,"taxa do banco (274), imersão (124), custo de projeto (119)",resposta,categoria taxa do banco 274 imersão 124 custo de projeto ...
5,6,filtro_periodo,78,resposta,78
6,7,filtro_periodo,"R$ 9.471,49, categoria ""projeto"", ENTRADA, gestão 18_19",resposta,"valor 9471.49 categoria projeto Name: 56, dtype: object"
7,8,calculo_derivado,"-R$ 32.301,25 (entrada R$ 14.520,00; saída R$ 46.821,25)",resposta,32301.25
8,9,calculo_derivado,"R$ 18.201,72",resposta,18201.72
9,10,calculo_derivado,8,resposta,8


In [6]:
for i in [7, 10, 12]:  # linhas 8, 11 e 13 (índice começa em 0)
    print(resultados_df.loc[i, "id"], "->", repr(resultados_df.loc[i, "codigo_gerado"]))

8 -> "df[df['gestao'] == '22'].groupby('tipo')['valor'].sum().diff().iloc[-1]"
11 -> "df[df['cliente_projeto'] == 'Cliente 001'].loc[df['tipo'] == 'SAÍDA', 'valor'].sum()"
13 -> None


Confira linha por linha contra `resposta_esperada` e preencha abaixo. `tipo_obtido` já mostra
se o agente tratou como resposta normal, `sem_resposta`, `recusa` ou `erro` — as perguntas
14 a 18 do gabarito **devem** vir como `sem_resposta` ou `recusa`, não como `resposta`.

In [ ]:
# Preencha manualmente após conferir cada linha acima (True/False)
acertos = [
    # id 1..20, na ordem -- ajuste conforme a conferência
]
if acertos:
    print(f"Acertou {sum(acertos)} de {len(acertos)}")